In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/nicher92/swedish_correctness_classifier.git

import json
from pathlib import Path

data_path = Path(
    "/content/swedish_correctness_classifier/data/train.jsonl"
)

with open(data_path, "r", encoding="utf-8") as f:
    train_data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Total training examples:", len(train_data))
print("\nFirst example:")
print(train_data[0])

fatal: destination path 'swedish_correctness_classifier' already exists and is not an empty directory.
Total training examples: 478

First example:
{'id': 2, 'text': 'De sålde deras hus förra året.', 'score': 0.45, 'label': 'poor', 'error': "wrong pronoun form ('sitt' -> 'deras')", 'group': 434}


In [ ]:
import pandas as pd

df = pd.DataFrame(train_data)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nScore statistics:")
print(df["score"].describe())

print("\nExamples:")
display(df[["text", "score", "label", "error"]].head(10))

Dataset shape: (478, 6)

Columns: ['id', 'text', 'score', 'label', 'error', 'group']

Label distribution:
label
poor    239
good    239
Name: count, dtype: int64

Score statistics:
count    478.000000
mean       0.651674
std        0.310470
min        0.000000
25%        0.400000
50%        0.675000
75%        0.950000
max        1.000000
Name: score, dtype: float64

Examples:


,text,score,label,error
0,De sålde deras hus förra året.,0.45,poor,wrong pronoun form ('sitt' -> 'deras')
1,Vi måste vänta för bussen i tjugo minuter.,0.40,poor,wrong preposition (vänta på)
2,Trots att vädret var dåligt kom nästan alla ti...,0.95,good,None
3,Om symtomen inte går över inom en vecka bör du...,0.95,good,None
4,Förra året reste vi till Japan.,0.95,good,None
5,Företaget har anställde tjugo nya medarbetarer...,0.20,poor,wrong verb form ('anställt' -> 'anställde'); w...
6,"Efter jobbet gick jag till gymmet, men det var...",0.95,good,None
7,Jag bor på Sverige.,0.40,poor,wrong preposition with country (i Sverige)
8,Hon köpte en ny vinter jacka.,0.45,poor,compound written as two words ('vinterjacka' -...
9,Ett barn satt på bänken och grät.,0.95,good,None


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path
import json
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(df, groups=df["group"])
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

train_groups = set(train_df["group"])
val_groups = set(val_df["group"])

assert train_groups.isdisjoint(val_groups)
print("Group leakage: None")

output_dir = Path("/content/grammar_data")
output_dir.mkdir(exist_ok=True)

for name, subset in [
    ("train", train_df),
    ("validation", val_df)
]:
    path = output_dir / f"{name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for row in subset.to_dict("records"):
            f.write(
                json.dumps(row, ensure_ascii=False) + "\n"
            )

print("Dataset files saved successfully!")

Training examples: 382
Validation examples: 96
Group leakage: None
Dataset files saved successfully!


In [ ]:
from pathlib import Path
import requests
import zipfile

url = (
    "https://svn.spraakbanken.gu.se/sb-arkiv/pub/"
    "dalaj/dalaj-ged-superlim/dalaj-ged-superlim.zip"
)

folder = Path("/content/dalaj_data")
folder.mkdir(exist_ok=True)

zip_path = folder / "dalaj.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()
zip_path.write_bytes(response.content)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(folder)
    print("Downloaded successfully!")
    print("\nFiles inside the dataset:")
    for name in archive.namelist():
        print(name)

Downloaded successfully!

Files inside the dataset:
dalaj-ged-superlim/
dalaj-ged-superlim/dalaj-ged-superlim_dev.jsonl
dalaj-ged-superlim/dalaj-ged-superlim_documentation-sheet.tsv
dalaj-ged-superlim/dalaj-ged-superlim_test.jsonl
dalaj-ged-superlim/dalaj-ged-superlim_train.jsonl


In [ ]:
import json
from pathlib import Path

dataset_dir = Path(
    "/content/dalaj_data/dalaj-ged-superlim"
)

for split in ["train", "dev", "test"]:
    path = dataset_dir / f"dalaj-ged-superlim_{split}.jsonl"

    with path.open("r", encoding="utf-8") as f:
        examples = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print(f"\n===== {split.upper()} =====")
    print("Total examples:", len(examples))

    for example in examples[:3]:
        print(json.dumps(
            example,
            ensure_ascii=False,
            indent=2
        ))


===== TRAIN =====
Total examples: 35581
{
  "sentence": "övrigt För övrigt avslutas texterna med en viktig visdom för livet.",
  "label": "incorrect",
  "meta": {
    "error_span": {
      "start": 0,
      "stop": 10
    },
    "confusion_pair": {
      "incorrect_span": "övrigt För",
      "correction": "Utöver detta"
    },
    "error_label": "L",
    "education_level": "Avancerad",
    "l1": "Arabiska",
    "data_source": "Dalaj.v.2 -- SweLL gold"
  }
}
{
  "sentence": "Övriga underverk är enligt läsarna Ishotellet i Jukkasjärvi, skyskrapan Turning Torso i Malmö, Oresundsbron och Globen-arenan (som under en kort period var världens största sfäriska byggnad).",
  "label": "correct",
  "meta": {
    "error_span": {
      "start": null,
      "stop": null
    },
    "confusion_pair": {
      "incorrect_span": null,
      "correction": null
    },
    "error_label": "",
    "education_level": "Avancerad",
    "l1": "",
    "data_source": "COCTAILL lesson text"
  }
}
{
  "sentence": "Ö

In [ ]:
from collections import Counter
from pathlib import Path
import json

dataset_dir = Path(
    "/content/dalaj_data/dalaj-ged-superlim"
)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

dalaj_train = load_jsonl(
    dataset_dir / "dalaj-ged-superlim_train.jsonl"
)

dalaj_dev = load_jsonl(
    dataset_dir / "dalaj-ged-superlim_dev.jsonl"
)

print("TRAINING LABELS:")
print(Counter(x["label"] for x in dalaj_train))

print("\nVALIDATION LABELS:")
print(Counter(x["label"] for x in dalaj_dev))

print("\nGRAMMAR ERROR TYPES:")
errors = Counter(
    x["meta"].get("error_label", "")
    for x in dalaj_train
    if x["label"] == "incorrect"
)

print(errors.most_common(20))

print("\nEXAMPLE INCORRECT SENTENCES:")
for example in [
    x for x in dalaj_train
    if x["label"] == "incorrect"
][:5]:
    print("\nSentence:", example["sentence"])
    print(
        "Error:",
        example["meta"]["confusion_pair"]
    )

TRAINING LABELS:
Counter({'incorrect': 18109, 'correct': 17472})

VALIDATION LABELS:
Counter({'correct': 2424, 'incorrect': 2278})

GRAMMAR ERROR TYPES:
[('M', 5120), ('S', 3761), ('O', 3179), ('L', 3007), ('P', 1736), ('L, S', 520), ('M, O', 348), ('L, M', 174), ('M, S', 86), ('L, O', 76), ('O, S', 46), ('L, M, O', 20), ('L, O, S', 18), ('', 7), ('L, M, S', 4), ('M, O, S', 3), ('M, O, P', 1), ('P, S', 1), ('O, P', 1), ('TA BORT', 1)]

EXAMPLE INCORRECT SENTENCES:

Sentence: övrigt För övrigt avslutas texterna med en viktig visdom för livet.
Error: {'incorrect_span': 'övrigt För', 'correction': 'Utöver detta'}

Sentence: Östling skriver om politiker som tycker att obligatorisk språkundervisningen och prov på nynorska krävs för att skydda ett minoritetsspråk som har ett gammalt kulturarv knutet till sig.
Error: {'incorrect_span': 'språkundervisningen', 'correction': 'språkundervisning'}

Sentence: Östling skriver om politiker som tycker att obligatorisk språkundervisning och prov på nyn

In [ ]:
from collections import defaultdict

def normalize(text):
    return " ".join(text.split()).strip()

sentence_labels = defaultdict(set)

for example in dalaj_train:
    text = normalize(example["sentence"])
    sentence_labels[text].add(example["label"])

conflicting = {
    text: labels
    for text, labels in sentence_labels.items()
    if len(labels) > 1
}

train_texts = set(sentence_labels)

dev_texts = {
    normalize(example["sentence"])
    for example in dalaj_dev
}

print("Total training rows:", len(dalaj_train))
print("Unique training sentences:", len(train_texts))
print("Duplicate rows:", len(dalaj_train) - len(train_texts))
print("Conflicting labels:", len(conflicting))
print("Train/Dev exact overlap:", len(train_texts & dev_texts))

Total training rows: 35581
Unique training sentences: 35462
Duplicate rows: 119
Conflicting labels: 0
Train/Dev exact overlap: 10


In [ ]:
import json
from pathlib import Path

def normalize(text):
    return " ".join(text.split()).strip()

# Original development sentences
dev_texts = {
    normalize(row["sentence"])
    for row in dalaj_dev
}

# Clean training sentences
seen = set()
clean_train = []

for row in dalaj_train:
    sentence = normalize(row["sentence"])

    if not sentence:
        continue

    if sentence in seen:
        continue

    if sentence in dev_texts:
        continue

    seen.add(sentence)

    clean_train.append({
        "text": sentence,
        "label": 1 if row["label"] == "correct" else 0,
        "source": "DaLAJ-GED-SuperLim"
    })

print("Original examples:", len(dalaj_train))
print("Clean training examples:", len(clean_train))
print("Removed examples:", len(dalaj_train) - len(clean_train))

# Save cleaned data
output_dir = Path("/content/grammar_data")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "dalaj_train_clean.jsonl"

with output_path.open("w", encoding="utf-8") as f:
    for row in clean_train:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_path)

Original examples: 35581
Clean training examples: 35452
Removed examples: 129
Saved: /content/grammar_data/dalaj_train_clean.jsonl


In [ ]:
from collections import Counter
from pathlib import Path
import json

# Check the cleaned class distribution
counts = Counter(row["label"] for row in clean_train)

print("Correct sentences:", counts[1])
print("Incorrect sentences:", counts[0])
print("Total:", len(clean_train))

# Read AI Sweden's official test sentences
test_path = Path(
    "/content/swedish_correctness_classifier/data/test.jsonl"
)

with test_path.open(encoding="utf-8") as f:
    ai_test = [json.loads(line) for line in f if line.strip()]

test_texts = {
    normalize(row["text"])
    for row in ai_test
}

# Remove exact overlap with AI Sweden test data
filtered_train = [
    row for row in clean_train
    if normalize(row["text"]) not in test_texts
]

print("\nAI Sweden test overlaps removed:",
      len(clean_train) - len(filtered_train))

print("Final training examples:", len(filtered_train))

# Save the final filtered dataset
path = Path("/content/grammar_data/dalaj_train_final.jsonl")

with path.open("w", encoding="utf-8") as f:
    for row in filtered_train:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved successfully:", path)

Correct sentences: 17435
Incorrect sentences: 18017
Total: 35452

AI Sweden test overlaps removed: 0
Final training examples: 35452
Saved successfully: /content/grammar_data/dalaj_train_final.jsonl


In [ ]:
!pip install -q "transformers==4.57.3" "scikit-learn==1.7.2"

import torch
import random
import numpy as np
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "KB/bert-base-swedish-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Device:", device)
print("Model:", MODEL_ID)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at KB/bert-base-swedish-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda
Model: KB/bert-base-swedish-cased
Parameters: 124692482


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class GrammarDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]

        encoded = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoded.items()
        }

        item["labels"] = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return item

train_dataset = GrammarDataset(
    filtered_train, tokenizer
)

dev_rows = [
    {
        "text": row["sentence"],
        "label": 1 if row["label"] == "correct" else 0
    }
    for row in dalaj_dev
]

val_dataset = GrammarDataset(
    dev_rows, tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Training batches:", len(train_loader))

Training examples: 35452
Validation examples: 4702
Training batches: 4432


In [ ]:
import torch
import random
import numpy as np
from torch.optim import AdamW
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score
from pathlib import Path

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Training configuration
EPOCHS = 1
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

# Mixed precision reduces GPU memory usage
scaler = torch.amp.GradScaler("cuda")

print("Starting Swedish BERT training...")
print("GPU:", torch.cuda.get_device_name(0))

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        progress.set_postfix(
            loss=round(loss.item(), 4)
        )

    print(
        "Average training loss:",
        total_loss / len(train_loader)
    )

print("Training completed!")

Starting Swedish BERT training...
GPU: Tesla T4


Epoch 1/1:   0%|          | 0/4432 [00:00<?, ?it/s]

Average training loss: 0.40966199435266776
Training completed!


In [ ]:
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score
)

model.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating"):
        labels = batch["labels"]

        inputs = {
            key: value.to("cuda")
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=-1
        )[:, 1]

        all_probs.extend(probabilities.cpu().tolist())
        all_labels.extend(labels.tolist())

predictions = [
    int(prob >= 0.5)
    for prob in all_probs
]

print("Validation accuracy:",
      round(accuracy_score(all_labels, predictions), 4))

print("Validation F1:",
      round(f1_score(all_labels, predictions), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(all_labels, all_probs), 4))

Validating:   0%|          | 0/294 [00:00<?, ?it/s]

Validation accuracy: 0.8305
Validation F1: 0.8462
Validation ROC-AUC: 0.922


In [ ]:
from pathlib import Path

save_path = Path("/content/swedish_bert_stage1")

model.save_pretrained(
    save_path,
    safe_serialization=True
)

tokenizer.save_pretrained(save_path)

print("Model saved successfully!")
print("Location:", save_path)

Model saved successfully!
Location: /content/swedish_bert_stage1


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

print("Drive connected:", Path("/content/drive/MyDrive").exists())
print("Model available:", Path("/content/swedish_bert_stage1/model.safetensors").exists())

Drive connected: False
Model available: False


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving swedish_bert_stage1 (1).zip to swedish_bert_stage1 (1).zip


In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/swedish_bert_stage1 (1).zip")
extract_dir = Path("/content/restored_model")

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_dir)

# Find the trained model
matches = list(extract_dir.rglob("model.safetensors"))

assert matches, "Model weights not found!"

MODEL_PATH = str(matches[0].parent)

print("Model restored successfully!")
print("Model location:", MODEL_PATH)

Model restored successfully!
Model location: /content/restored_model/swedish_bert_stage1


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda


In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

repo = Path("/content/swedish_correctness_classifier")

if not (repo / "data/train.jsonl").exists():
    import subprocess
    subprocess.run([
        "git", "clone",
        "https://github.com/nicher92/swedish_correctness_classifier.git",
        str(repo)
    ], check=True)

with open(repo / "data/train.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(df, groups=df["group"])
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print(
    "Group leakage:",
    len(set(train_df["group"]) & set(val_df["group"]))
)

Training examples: 382
Validation examples: 96
Group leakage: 0


In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm

# Confirm that class 1 means grammatically correct
print("Model labels:", model.config.id2label)

def predict_scores(texts, batch_size=32):
    scores = []
    model.eval()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)

            probabilities = torch.softmax(
                outputs.logits.float(),
                dim=-1
            )[:, 1]

        scores.extend(
            probabilities.cpu().tolist()
        )

    return np.array(scores)

# Test some Swedish sentences
examples = [
    "Jag har köpt en ny bil.",
    "Jag har köpa en ny bil.",
    "Hon går till skolan.",
    "Hon gå till skolan."
]

scores = predict_scores(examples)

for sentence, score in zip(examples, scores):
    print(f"{score:.4f} | {sentence}")

Model labels: {0: 'LABEL_0', 1: 'LABEL_1'}


  0%|          | 0/1 [00:00<?, ?it/s]

0.9841 | Jag har köpt en ny bil.
0.0313 | Jag har köpa en ny bil.
0.9820 | Hon går till skolan.
0.0275 | Hon gå till skolan.


In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
import numpy as np
import joblib

# Generate probabilities on the training subset
train_prob = predict_scores(
    train_df["text"].tolist()
)

train_targets = train_df["score"].to_numpy(dtype=float)
val_targets = val_df["score"].to_numpy(dtype=float)

# Fit score calibration using training data only
calibrator = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    out_of_bounds="clip"
)

calibrator.fit(train_prob, train_targets)

# Use the validation predictions from Step 20
calibrated_scores = calibrator.predict(validation_scores)

print("BEFORE CALIBRATION")
print("Spearman:", round(
    spearmanr(val_targets, validation_scores).statistic, 4
))
print("MAE:", round(
    mean_absolute_error(val_targets, validation_scores), 4
))

print("\nAFTER CALIBRATION")
print("Spearman:", round(
    spearmanr(val_targets, calibrated_scores).statistic, 4
))
print("MAE:", round(
    mean_absolute_error(val_targets, calibrated_scores), 4
))
print("ROC-AUC:", round(
    roc_auc_score(
        (val_df["label"] == "good").astype(int),
        calibrated_scores
    ), 4
))

joblib.dump(
    calibrator,
    "/content/restored_model/calibrator.joblib"
)

print("\nCalibrator saved successfully!")

  0%|          | 0/12 [00:00<?, ?it/s]

BEFORE CALIBRATION
Spearman: 0.7483
MAE: 0.211

AFTER CALIBRATION
Spearman: 0.7498
MAE: 0.163
ROC-AUC: 0.9138

Calibrator saved successfully!


In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, mean_absolute_error

validation_scores = predict_scores(
    val_df["text"].tolist()
)

true_scores = val_df["score"].to_numpy(dtype=float)

spearman = spearmanr(
    true_scores, validation_scores
).statistic

auc = roc_auc_score(
    (val_df["label"] == "good").astype(int),
    validation_scores
)

mae = mean_absolute_error(
    true_scores, validation_scores
)

print("Spearman:", round(spearman, 4))
print("ROC-AUC:", round(auc, 4))
print("MAE:", round(mae, 4))

  0%|          | 0/3 [00:00<?, ?it/s]

Spearman: 0.7483
ROC-AUC: 0.918
MAE: 0.211


In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
from pathlib import Path
import joblib

# Generate predictions on AI Sweden training data
train_prob = predict_scores(train_df["text"].tolist())

train_targets = train_df["score"].to_numpy(dtype=float)
val_targets = val_df["score"].to_numpy(dtype=float)

# Train calibrator on training examples only
calibrator = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    out_of_bounds="clip"
)

calibrator.fit(train_prob, train_targets)

# Calibrate validation predictions
calibrated_scores = calibrator.predict(validation_scores)

print("BEFORE CALIBRATION")
print("Spearman:", round(
    spearmanr(val_targets, validation_scores).statistic, 4
))
print("MAE:", round(
    mean_absolute_error(val_targets, validation_scores), 4
))

print("\nAFTER CALIBRATION")
print("Spearman:", round(
    spearmanr(val_targets, calibrated_scores).statistic, 4
))
print("ROC-AUC:", round(
    roc_auc_score(
        (val_df["label"] == "good").astype(int),
        calibrated_scores
    ), 4
))
print("MAE:", round(
    mean_absolute_error(val_targets, calibrated_scores), 4
))

# Save the calibration model
save_path = Path("/content/swedish_calibrator.joblib")
joblib.dump(calibrator, save_path)

print("\nCalibrator saved:", save_path)

  0%|          | 0/12 [00:00<?, ?it/s]

BEFORE CALIBRATION
Spearman: 0.7483
MAE: 0.211

AFTER CALIBRATION
Spearman: 0.7498
ROC-AUC: 0.9138
MAE: 0.163

Calibrator saved: /content/swedish_calibrator.joblib


In [ ]:
from google.colab import files

files.download("/content/swedish_calibrator.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from pathlib import Path

results = {
    "model": "KB/bert-base-swedish-cased",
    "pretraining_data": "DaLAJ-GED-SuperLim",
    "stage1_training_examples": 35452,
    "stage2_calibration_examples": 382,
    "validation_examples": 96,
    "raw": {
        "spearman": 0.7483,
        "roc_auc": 0.9180,
        "mae": 0.2110
    },
    "calibrated": {
        "spearman": 0.7498,
        "roc_auc": 0.9138,
        "mae": 0.1630
    }
}

path = Path("/content/calibration_results.json")

with path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Results saved:", path)

Results saved: /content/calibration_results.json


In [ ]:
from google.colab import files

files.download("/content/calibration_results.json")
files.download("/content/swedish_calibrator.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
import random
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reload original Stage 1 weights
# This keeps our calibrated classifier unchanged.
reg_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
).to(device)

# Freeze the pretrained encoder for the first experiment
for parameter in reg_model.bert.parameters():
    parameter.requires_grad = False

# Train only the output classification layer
for parameter in reg_model.classifier.parameters():
    parameter.requires_grad = True

print("Regression experiment ready!")
print("Device:", device)

print(
    "Trainable parameters:",
    sum(p.numel() for p in reg_model.parameters()
        if p.requires_grad)
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Regression experiment ready!
Device: cuda
Trainable parameters: 1538


In [ ]:
class ScoreDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text"].tolist()
        self.scores = dataframe["score"].astype(float).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        item = {
            k: v.squeeze(0)
            for k, v in encoded.items()
        }

        item["score"] = torch.tensor(
            self.scores[index],
            dtype=torch.float32
        )

        return item

reg_train_dataset = ScoreDataset(train_df, tokenizer)
reg_val_dataset = ScoreDataset(val_df, tokenizer)

reg_train_loader = DataLoader(
    reg_train_dataset,
    batch_size=16,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

reg_val_loader = DataLoader(
    reg_val_dataset,
    batch_size=32,
    shuffle=False
)

print("Training examples:", len(reg_train_dataset))
print("Validation examples:", len(reg_val_dataset))
print("Training batches:", len(reg_train_loader))

Training examples: 382
Validation examples: 96
Training batches: 24


In [ ]:
import copy

from torch.optim import AdamW
from sklearn.metrics import mean_absolute_error

EPOCHS = 8
LEARNING_RATE = 1e-3

optimizer = AdamW(
    reg_model.classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

best_mae = float("inf")
best_state = None

for epoch in range(EPOCHS):

    reg_model.train()
    total_loss = 0

    for batch in reg_train_loader:
        targets = batch.pop("score").to(device)

        inputs = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = reg_model(**inputs)

        # Probability of grammatically correct Swedish
        scores = torch.softmax(
            outputs.logits.float(), dim=-1
        )[:, 1]

        loss = torch.nn.functional.mse_loss(
            scores, targets
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    reg_model.eval()

    predicted = []
    actual = []

    with torch.no_grad():
        for batch in reg_val_loader:
            targets = batch.pop("score")

            inputs = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = reg_model(**inputs)

            scores = torch.softmax(
                outputs.logits.float(), dim=-1
            )[:, 1]

            predicted.extend(scores.cpu().tolist())
            actual.extend(targets.tolist())

    val_mae = mean_absolute_error(actual, predicted)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train loss: {total_loss / len(reg_train_loader):.4f} | "
        f"Validation MAE: {val_mae:.4f}"
    )

    if val_mae < best_mae:
        best_mae = val_mae
        best_state = copy.deepcopy(reg_model.state_dict())

# Restore best validation checkpoint
reg_model.load_state_dict(best_state)
reg_model.eval()

print("\nBest validation MAE:", round(best_mae, 4))
print("Regression fine-tuning completed!")

Epoch 1/8 | Train loss: 0.0559 | Validation MAE: 0.1451
Epoch 2/8 | Train loss: 0.0417 | Validation MAE: 0.1430
Epoch 3/8 | Train loss: 0.0357 | Validation MAE: 0.1442
Epoch 4/8 | Train loss: 0.0375 | Validation MAE: 0.1413
Epoch 5/8 | Train loss: 0.0360 | Validation MAE: 0.1387
Epoch 6/8 | Train loss: 0.0334 | Validation MAE: 0.1390
Epoch 7/8 | Train loss: 0.0361 | Validation MAE: 0.1566
Epoch 8/8 | Train loss: 0.0357 | Validation MAE: 0.1337

Best validation MAE: 0.1337
Regression fine-tuning completed!


In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import (
    mean_absolute_error,
    roc_auc_score
)

reg_model.eval()

reg_scores = []
reg_targets = []

with torch.no_grad():
    for batch in reg_val_loader:
        targets = batch.pop("score")

        inputs = {
            k: v.to(device)
            for k, v in batch.items()
        }

        logits = reg_model(**inputs).logits.float()
        scores = torch.softmax(logits, dim=-1)[:, 1]

        reg_scores.extend(scores.cpu().tolist())
        reg_targets.extend(targets.tolist())

reg_spearman = spearmanr(
    reg_targets, reg_scores
).statistic

reg_auc = roc_auc_score(
    (val_df["label"] == "good").astype(int),
    reg_scores
)

reg_mae = mean_absolute_error(
    reg_targets, reg_scores
)

print("REGRESSION FINE-TUNING RESULTS")
print("Spearman:", round(reg_spearman, 4))
print("ROC-AUC:", round(reg_auc, 4))
print("MAE:", round(reg_mae, 4))

print("\nPREVIOUS CALIBRATED MODEL")
print("Spearman: 0.7498")
print("ROC-AUC: 0.9138")
print("MAE: 0.1630")

REGRESSION FINE-TUNING RESULTS
Spearman: 0.7559
ROC-AUC: 0.9314
MAE: 0.1337

PREVIOUS CALIBRATED MODEL
Spearman: 0.7498
ROC-AUC: 0.9138
MAE: 0.1630


In [ ]:
from pathlib import Path

reg_save_path = Path("/content/swedish_bert_regression")

reg_model.save_pretrained(
    reg_save_path,
    safe_serialization=True
)

tokenizer.save_pretrained(reg_save_path)

print("Regression model saved:", reg_save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Regression model saved: /content/swedish_bert_regression


In [ ]:
import torch
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, mean_absolute_error

reg_model.eval()

reg_scores = []

with torch.inference_mode():
    for batch in reg_val_loader:
        inputs = {
            k: v.to(device)
            for k, v in batch.items()
            if k != "score"
        }

        logits = reg_model(**inputs).logits.float()
        probabilities = torch.softmax(logits, dim=-1)[:, 1]

        reg_scores.extend(probabilities.cpu().tolist())

actual_scores = val_df["score"].to_numpy(dtype=float)
actual_labels = (val_df["label"] == "good").astype(int)

print("REGRESSION MODEL RESULTS")
print("Spearman:", round(
    spearmanr(actual_scores, reg_scores).statistic, 4
))
print("ROC-AUC:", round(
    roc_auc_score(actual_labels, reg_scores), 4
))
print("MAE:", round(
    mean_absolute_error(actual_scores, reg_scores), 4
))

REGRESSION MODEL RESULTS
Spearman: 0.7559
ROC-AUC: 0.9314
MAE: 0.1337


In [ ]:
from pathlib import Path
import shutil

model_dir = Path("/content/swedish_bert_regression")

assert (model_dir / "model.safetensors").exists(), (
    "Regression model weights not found"
)

zip_path = shutil.make_archive(
    "/content/swedish_bert_regression",
    "zip",
    root_dir="/content",
    base_dir="swedish_bert_regression"
)

print("ZIP created:", zip_path)
print("Size (MB):", round(Path(zip_path).stat().st_size / 1e6, 1))

ZIP created: /content/swedish_bert_regression.zip
Size (MB): 463.3


In [ ]:
from google.colab import files

files.download("/content/swedish_bert_regression.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from pathlib import Path

experiment = {
    "model": "KB/bert-base-swedish-cased",
    "training": {
        "stage1_dataset": "DaLAJ-GED-SuperLim",
        "stage1_examples": 35452,
        "stage2_dataset": "AI Sweden starter training split",
        "stage2_examples": 382,
        "stage2_epochs": 8,
        "seed": 42
    },
    "validation": {
        "examples": 96,
        "spearman": 0.7559,
        "roc_auc": 0.9314,
        "mae": 0.1337
    }
}

results_path = Path("/content/regression_results.json")

with results_path.open("w", encoding="utf-8") as f:
    json.dump(experiment, f, indent=2)

print("Results saved:", results_path)

Results saved: /content/regression_results.json


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

model_dir = Path("/content/swedish_bert_regression")

assert (model_dir / "model.safetensors").exists(), \
    "Regression model not found"

zip_path = shutil.make_archive(
    "/content/swedish_bert_regression",
    "zip",
    root_dir="/content",
    base_dir="swedish_bert_regression"
)

print("Backup created:", zip_path)

files.download(zip_path)

Backup created: /content/swedish_bert_regression.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("/content/swedish_bert_regression.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("/content/regression_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
{"id": 1, "text": "Jag har köpt en ny bil."}

{'id': 1, 'text': 'Jag har köpt en ny bil.'}

In [ ]:
import torch

sentence = "Jag har köpt en ny bil."

reg_model.eval()

inputs = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

with torch.inference_mode():
    logits = reg_model(**inputs).logits.float()
    score = torch.softmax(logits, dim=-1)[0, 1].item()

print({"id": 1, "score": round(score, 4)})

{'id': 1, 'score': 0.9173}


In [ ]:
python predict.py --input test.jsonl --output scores.jsonl

SyntaxError: invalid syntax (2933251006.py, line 1)

In [ ]:
%%writefile /content/predict.py
import argparse
import json
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

MODEL_ID = "swedish_bert_regression"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    model_path = Path(__file__).parent / MODEL_ID

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.to(device)
    model.eval()

    records = []

    with open(args.input, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    results = []

    for start in range(0, len(records), 32):
        batch = records[start:start + 32]
        texts = [record["text"] for record in batch]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)

        with torch.inference_mode():
            logits = model(**inputs).logits.float()
            scores = torch.softmax(logits, dim=-1)[:, 1]
            scores = scores.clamp(0, 1).cpu().tolist()

        for record, score in zip(batch, scores):
            results.append({
                "id": record["id"],
                "score": float(score),
            })

    with open(args.output, "w", encoding="utf-8") as f:
        for result in results:
            f.write(
                json.dumps(result, ensure_ascii=False) + "\n"
            )


if __name__ == "__main__":
    main()

Writing /content/predict.py


In [ ]:
import json

test_records = [
    {"id": 1, "text": "Jag har köpt en ny bil."},
    {"id": 2, "text": "Jag har köpa en ny bil."},
]

with open("/content/test_input.jsonl", "w", encoding="utf-8") as f:
    for record in test_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Test input created!")

Test input created!


In [ ]:
!python /content/predict.py --input /content/test_input.jsonl --output /content/test_output.jsonl

with open("/content/test_output.jsonl", encoding="utf-8") as f:
    print(f.read())

Loading weights: 100% 201/201 [00:00<00:00, 8785.12it/s]
{"id": 1, "score": 0.9172777533531189}
{"id": 2, "score": 0.2829154431819916}



In [ ]:
import subprocess
from pathlib import Path

repo = Path("/content/swedish_correctness_classifier")

if not (repo / "evaluate.py").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/nicher92/swedish_correctness_classifier.git",
            str(repo)
        ],
        check=True
    )

result = subprocess.run(
    [
        "python",
        str(repo / "evaluate.py"),
        "--repo",
        "/content"
    ],
    cwd=str(repo),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


120 test examples, timing covers the whole predict.py run including model loading

| Spearman | ROC-AUC | Time (s) | Samples/min |
|---|---|---|---|
| 0.821 | 0.963 | 17.8 | 406 |


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4249.68it/s]

Exit code: 0
